# Principle of Virtual Work — symbolic derivation

Starting point (PVW in d'Alembert form; LHS − RHS = 0 is implied):

$$\int_V \boldsymbol{\sigma}:\nabla\delta\mathbf{u}\,dV
  - \int_V \mathbf{f}\cdot\delta\mathbf{u}\,dV
  - \int_{\partial V} \mathbf{t}\cdot\delta\mathbf{u}\,dS
  + \int_V \rho\ddot{\mathbf{u}}\cdot\delta\mathbf{u}\,dV = 0$$

We apply integration by parts, collect terms by virtual displacement $\delta\mathbf{u}$,
and localize to recover the **strong-form equilibrium equation** and the **traction BC**.

> **Prerequisite:** build the Python bindings and set `PYTHONPATH`, or run
> `source examples/env.sh` before launching Jupyter.

In [ ]:
from IPython.display import display, Math, Latex

from tender import (
    tensor,
    make_volume_domain,
    integral,
    ddot,
    dot,
    gradient,
    State,
    Derivation,
    apply_integration_by_parts_step,
    collect_step,
    localize_step,
    show_jupyter,
    to_latex_document,
    Sum, Scale, Contract, Divergence,
)

## Named tensors and geometry

In [ ]:
sigma   = tensor(r"\boldsymbol{\sigma}", 2)       # Cauchy stress
delta_u = tensor(r"\delta\mathbf{u}",    1)        # virtual displacement
f_body  = tensor(r"\mathbf{f}",          1)        # body force
t_trac  = tensor(r"\mathbf{t}",          1)        # surface traction
rho_udd = tensor(r"\rho\ddot{\mathbf{u}}", 1)     # inertia force ρü
n       = tensor(r"\mathbf{n}",          1)        # outward unit normal

V  = make_volume_domain("V", n)
dV = V.surface_boundary                             # ∂V

## PVW expression

In [ ]:
pvw = (
      integral(V,  ddot(sigma,  gradient(delta_u)))
    - integral(V,  dot(f_body,  delta_u))
    - integral(dV, dot(t_trac,  delta_u))
    + integral(V,  dot(rho_udd, delta_u))
)
display(Math(pvw.latex()))

## Step 1: Integration by parts

Apply IBP to $\int_V \boldsymbol{\sigma}:\nabla\delta\mathbf{u}\,dV$, yielding a surface term
via the divergence theorem.

In [ ]:
ibp_history = Derivation([
    apply_integration_by_parts_step(V),
]).apply(State(pvw))

show_jupyter(ibp_history)

## Step 2: Collect terms by $\delta\mathbf{u}$

Group integrals by domain — surface terms under $\int_{\partial V}$, volume terms under $\int_V$.

In [ ]:
collected_history = Derivation([
    collect_step(delta_u),
]).apply(ibp_history[-1])

show_jupyter(collected_history[1:])

## Localize over $V$ — equilibrium equation

The integral vanishes for all admissible $\delta\mathbf{u}$, so the integrand must vanish pointwise
(fundamental lemma of the calculus of variations).

In [ ]:
vol_history = Derivation([
    localize_step(V),
]).apply(collected_history[-1])

show_jupyter(vol_history[1:])

## Localize over $\partial V$ — traction boundary condition

In [ ]:
srf_history = Derivation([
    localize_step(dV),
]).apply(collected_history[-1])

show_jupyter(srf_history[1:])

## Verification

In [ ]:
vol_result = vol_history[-1].expr

assert isinstance(vol_result, Contract), \
    f"Expected Contract after volume localization, got {type(vol_result).__name__}"

def contains_divergence(e):
    if isinstance(e, Divergence): return True
    if isinstance(e, Scale):      return contains_divergence(e.expr)
    if isinstance(e, Sum):        return any(contains_divergence(t) for t in e.terms)
    if isinstance(e, Contract):   return contains_divergence(e.lhs) or contains_divergence(e.rhs)
    return False

assert contains_divergence(vol_result.lhs), \
    "∇·σ term not found in volume equilibrium equation"

print("Derivation verified:")
print("  • IBP applied correctly")
print("  • Divergence of sigma appears in the volume equilibrium equation")
print("  • Surface traction BC recovered via ∂V localization")

## Export to PDF

Write a compilable LaTeX document to `out/pvw_continuum.tex`.
Or simply run `make pvw_continuum` from the `examples/` directory.

In [ ]:
import pathlib

full_history = ibp_history + collected_history[1:] + vol_history[1:] + srf_history[1:]
tex = to_latex_document(
    full_history,
    title="Principle of Virtual Work — symbolic derivation",
)
out_dir = pathlib.Path("out")
out_dir.mkdir(exist_ok=True)
(out_dir / "pvw_continuum.tex").write_text(tex)
print("Written to out/pvw_continuum.tex")
print("Compile with: pdflatex -output-directory out out/pvw_continuum.tex")